Import relevant libraries:

In [ ]:
import numpy as np, matplotlib.pyplot as plt, scipy as sci, gzip
from scipy import special

Load files:

In [230]:
with gzip.open(r"C:\Users\629284\OneDrive - TGS Kew\Desktop\Python Projects\Python\Neural Net\data\MNIST\raw\t10k-images-idx3-ubyte.gz", "rb") as f:
    data = np.frombuffer(f.read(), dtype=np.uint8, offset=16)
    images = data.reshape(-1, 28, 28)
    flattened = images.reshape(-1, 784)

with gzip.open(r"C:\Users\629284\OneDrive - TGS Kew\Desktop\Python Projects\Python\Neural Net\data\MNIST\raw\t10k-labels-idx1-ubyte.gz", "rb") as l:
    labels = np.frombuffer(l.read(), dtype=np.uint8, offset=8)

Define activation function and neural net classes:

In [ ]:
def sigmoid(x):
    #return 1 / (1 + np.exp(-x))
    return sci.special.expit(x)

class Layer:
    def __init__(self, input_size : int, output_size : int):
        self.input_size = input_size
        self.output_size = output_size
        
        # Initialise weights with random values
        self.weights = np.random.randn(output_size, input_size) * np.sqrt(1 / input_size)
        self.biases = np.zeros(output_size)

    def get_activation(self, input: np.ndarray):
        if input.size != self.input_size:
            raise Exception(f"Error: Input size ({input.size}) does not match expected input size ({self.input_size})!")
        
        # Return both the sigmoided values and z values.
        weighted_value = np.matmul(self.weights, input)
        biased_value = weighted_value + self.biases
        return [sigmoid(biased_value), biased_value]

class Network:
    def __init__(self, input_layer_size: int, output_layer_size: int, hidden_layers: list[int]):
        # Create input layer
        self.input_layer = np.zeros(input_layer_size)

        self.hidden_layers : list[Layer] = []
        prev_layer_input = input_layer_size
        
        # Create hidden layers
        for layer in hidden_layers:
            self.hidden_layers.append(Layer(prev_layer_input, layer))
            prev_layer_input = layer

        self.output_layer = Layer(prev_layer_input, output_layer_size)

    def forward_pass(self, input: np.ndarray):
        self.set_input_layer(input)
        activations : list = [self.input_layer]
        z_values : list = []
        
        # Hidden layers
        for layer in self.hidden_layers:
            current = layer.get_activation(activations[-1])
            activations.append(current[0])
            z_values.append(current[1])
        
        # Output layer
        current = self.output_layer.get_activation(activations[-1])
        activations.append(current[0])
        z_values.append(current[1])
        return [activations, z_values]

    def backpropagation(self, input : np.ndarray, label : int, learning_rate : float):
        activations, z_values = self.forward_pass(input)
        one_hot = np.zeros(10)
        one_hot[label] = 1
        
        # ======== OUTPUT LAYER ========
        output = activations[-1]
        
        # How wrong each neuron is * sigmoid sensitivity
        delta = (one_hot - output) * output * (1 - output)
        
        # Last hidden layer
        previous_activation = activations[-2]
        
        
        weight_gradient = np.outer(delta, previous_activation)
        
        # Update output layer
        self.output_layer.weights += learning_rate * weight_gradient
        self.output_layer.biases += learning_rate * delta
        
        # ======== HIDDEN LAYERS ========
        
        next_delta = delta
        next_weights = self.output_layer.weights
        
        for i in range(len(self.hidden_layers) - 1, -1, -1):
            
            # Activation of this layer
            activation = activations[i + 1]
            
            # Propagate "blame" backwards
            delta = np.matmul(np.transpose(next_weights), next_delta)
            
            # Apply sigmoid sensitivity
            delta = delta * activation * (1 - activation)
            
            # Activation of previous layer
            previous_activation = activations[i]
            
            # Gradient of this layers weights
            weight_gradient = np.outer(delta, previous_activation)
            
            # Update this layer
            self.hidden_layers[i].weights += learning_rate * weight_gradient
            self.hidden_layers[i].biases += learning_rate * delta
            
            # Prep for next layer
            next_delta = delta
            next_weights = self.hidden_layers[i].weights

    def get_prediction(self, input : np.ndarray):
        output = self.forward_pass(input)[0][-1]
        return(np.argmax(output))

    def set_input_layer(self, arr : np.ndarray):
        if arr.size != self.input_layer.size:
            raise Exception(f"Error: Input size ({arr.size}) does not match expected input size ({self.input_layer.size})!")
        self.input_layer = arr

    def get_loss(self, output : np.ndarray, label : int):
        one_hot = np.zeros(10, dtype=int)
        one_hot[label] = 1
        
        return np.mean((output-one_hot) ** 2)
    
    def train(batch : np.ndarray):
        
    
    def save(self, filename: str):
        data = {}
        
        for i, layer in enumerate(self.hidden_layers):
            data[f"hidden_{i}_weights"] = layer.weights
            data[f"hidden_{i}_biases"] = layer.biases
        
        data["output_weights"] = self.output_layer.weights
        data["output_biases"] = self.output_layer.biases
        
        np.savez(filename, **data)
    
    def load(self, filename):
        try:
            data = np.load(filename)
        except FileNotFoundError:
            print("Model file not found. Maybe you made a typo?")
            return
        
        for i, layer in enumerate(self.hidden_layers):
            layer.weights = data[f"hidden_{i}_weights"]
            layer.biases = data[f"hidden_{i}_biases"]
        
        self.output_layer.weights = data["output_weights"]
        self.output_layer.biases = data["output_biases"]

Do stuff with it now:


In [237]:
amount_right = 0
amount = 0
for i in range(len(flattened)):
    if net.get_prediction(flattened[i]) == labels[i]:
        amount_right += 1
    amount += 1

print(f"Accuracy: {amount_right/amount*100}%")

Accuracy: 79.54%
